# Mini Project 6 — IoT Device Events: Bronze → Silver

One messy CSV of IoT device events → a clean, typed Delta table.

**Techniques in this project:**
- Robust CSV read (`quote` / `escape` / `multiline`)
- Renaming messy headers in one pass (`withColumnsRenamed`)
- Exploration before cleaning: schema, counts, duplicates, nulls, categorical scan
- **Grain-aware deduplication** (the most important lesson here — see section 3)
- Timestamps arriving in **three different formats** → `coalesce(try_to_timestamp(...))`
- Placeholders (`N/A`, `UNKNOWN`, `ERROR`) → real `NULL`
- Dirty money text (`$`, `€`, comma decimals, `EUR` suffix) cleaned **before** casting to `decimal`
- Timezone normalization (CET → UTC) at the silver layer
- **Nested JSON in three shapes** — `struct`, `array<struct>`, `map` — parsed with `from_json`
- Cleaning a field *inside* a struct with `withField` (no flattening needed)
- Idempotent `overwrite` write

## 1 · Read the raw CSV

The JSON columns contain commas and quotes, so a naive CSV read would shred them.
`quote` + `escape` + `multiline` tell Spark how the messy cells are wrapped.

In [ ]:
iot_device_raw_df = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiline", True)
    .format("csv")
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/iot_device_events.csv")
)
iot_device_raw_df.display()

## 2 · Standardize column names

Raw headers have spaces, mixed case and units baked in (`Device ID`, `Temp_reading (C)`).
Renaming everything to `snake_case` once, up front, means no backtick-quoting later.

In [ ]:
iot_device_raw_df = iot_device_raw_df.withColumnsRenamed({
    "Device ID": "device_id",
    "event TS": "event_time",
    "Temp_reading (C)": "temp_reading",
    "Cost": "cost",
    "Firmware Ver": "firm_version",
    "device_meta": "device_info",
    "tags": "device_tags",
    "settings": "device_settings"
})
iot_device_raw_df.display()

## 3 · Explore before cleaning

Everything arrives as `string` (no schema was given) — that's fine for bronze;
we measure first, then fix.

In [ ]:
iot_device_raw_df.printSchema()

### Duplicate check — and a grain lesson

Three counts tell the whole story:

- total rows: **122**
- distinct `device_id`: **120**
- distinct **full rows**: **120**

So the 2 extra rows are **exact copies** of other rows.

**The trap:** this table is *event*-level (one row per device event), so
`dropDuplicates(["device_id"])` would be wrong in general — one device can
legitimately send many events, and a subset-dedup keeps an *arbitrary* row and
silently deletes the rest. It happens to give the same count on this data,
but "row counts match" ≠ "logic is right".

**The safe move for event data:** dedupe on the **full row** (exact duplicates only).

In [ ]:
print(iot_device_raw_df.count())                                  # total rows
print(iot_device_raw_df.select("device_id").distinct().count())   # distinct devices
print(iot_device_raw_df.distinct().count())                       # distinct full rows

In [ ]:
from pyspark.sql.functions import col

# key column must never be null
iot_device_raw_df.filter(col("device_id").isNull()).count()

### Categorical scan

`groupBy(...).count()` on low-cardinality columns exposes the placeholder values
(`N/A`, `UNKNOWN`, `ERROR`) and format variants (`V2.1` vs `v2.1`) we need to clean.

In [ ]:
iot_device_raw_df.groupBy("temp_reading").count().display()

In [ ]:
iot_device_raw_df.groupBy("firm_version").count().display()

## 4 · Clean the messy columns

All rules in one `withColumns` pass — each column gets its own treatment:

- **`event_time`** — arrives in 3 different formats. A strict parser would crash on the
  ones that don't match; `try_to_timestamp` returns `NULL` instead, and `coalesce`
  keeps whichever format worked.
- **`temp_reading`** — placeholders (`N/A`, `UNKNOWN`) become real `NULL`
  (a null-check can't see the *string* `"N/A"`).
- **`firm_version`** — strip the `V` / `v` prefix so versions compare consistently.
- **`cost`** — order matters: first placeholders → `NULL`, then strip currency symbols
  (`$`, `€`, ` EUR`) and fix the comma decimal separator. Clean the text **first**,
  cast **second**.
- **Dedup** — full-row only (exact copies), per the grain note above.

In [ ]:
from pyspark.sql.functions import try_to_timestamp, coalesce, lit, col, trim, when, replace

iot_clean_raw_df = iot_device_raw_df.withColumns({
    "event_time": coalesce(
        try_to_timestamp(trim(col("event_time")), lit("yyyy/MM/dd HH:mm:ss")),
        try_to_timestamp(trim(col("event_time")), lit("MM-dd-yyyy HH:mm:ss.SSS")),
        try_to_timestamp(trim(col("event_time")), lit("dd-MM-yyyy HH:mm:ss.SSSS")),
    ),
    "temp_reading": when(trim(col("temp_reading")).isin("N/A", "UNKNOWN"), None)
                    .otherwise(col("temp_reading")),
    "firm_version": replace(replace(trim(col("firm_version")), lit("V"), lit("")),
                            lit("v"), lit("")),
    "cost": when(trim(col("cost")).isin("N/A", "ERROR"), None)
            .otherwise(replace(
                replace(
                    replace(
                        replace(trim(col("cost")), lit("$"),    lit("")),
                                                   lit("€"),    lit("")),
                                                   lit(","),    lit(".")),
                                                   lit(" EUR"), lit(""))),
}).dropDuplicates()   # exact duplicates only — keeps the event grain intact

iot_clean_raw_df.display()

### Verify the timestamp parse

If a fourth, unexpected format existed, those rows would now be silently `NULL`.
Always check before moving on.

In [ ]:
iot_clean_raw_df.filter(col("event_time").isNull()).count()

### Cast to proper types

Text is clean now, so casting is safe. `decimal(10,2)` for money — exact,
unlike `double` which accumulates rounding errors.

In [ ]:
from pyspark.sql.functions import col

iot_clean_raw_df = iot_clean_raw_df.withColumns({
    "temp_reading": col("temp_reading").cast("double"),
    "cost": col("cost").cast("decimal(10,2)"),
})

iot_clean_raw_df.display()

### Business rule: negative cost → NULL

A negative cost is a valid *decimal* but an invalid *business value*.
Here we **keep the row and null the value**: the event itself is real — only its
cost is broken. (Different from dropping the whole row; which is right depends on
what downstream consumers need. Missing ≠ invalid.)

In [ ]:
from pyspark.sql.functions import col, when

iot_clean_df = iot_clean_raw_df.withColumn(
    "cost", when(col("cost") < 0, None).otherwise(col("cost"))
)
iot_clean_df.display()

## 5 · Normalize the timezone (CET → UTC)

Device timestamps arrive as naive CET local time. Converting once at the silver
layer means every downstream consumer reads **one consistent clock** — the standard
practice is to store UTC and convert to local time only for display.

In [ ]:
from pyspark.sql.functions import convert_timezone, lit

iot_clean_df = iot_clean_df.withColumns({
    "event_time": convert_timezone(lit("CET"), lit("UTC"), "event_time")
})

iot_clean_df.display()

## 6 · Parse the JSON columns

Three columns, three different JSON shapes — each needs a matching schema:

| Column | Shape | Schema | Why |
|---|---|---|---|
| `device_info` | fixed keys | `struct<...>` | same fields on every row |
| `device_tags` | repeated items | `array<struct<...>>` | 0..n sensor readings |
| `device_settings` | arbitrary key/value pairs | `map<string,string>` | keys differ per device |

In [ ]:
from pyspark.sql.functions import from_json

info_schema     = "struct<city string, country string, status string>"
tags_schema     = "array<struct<sensor string, value string>>"
settings_schema = "map<string, string>"

iot_final_df = (
    iot_clean_df.withColumns({
        "device_info":     from_json("device_info", info_schema),
        "device_tags":     from_json("device_tags", tags_schema),
        "device_settings": from_json("device_settings", settings_schema)
    })
)
iot_final_df.display()

### Clean a field *inside* the struct — `withField`

`device_info.status` still carries placeholders (`N/A`, `UNKNOWN`, `ERROR`).
`withField` rewrites **one field inside the struct in place** — no exploding,
no flattening-and-rebuilding. The struct stays intact, the dirty field becomes
a real `NULL` (and valid values get normalized with `lower` + `trim`).

In [ ]:
from pyspark.sql.functions import when, lower, trim, col

iot_final_df = iot_final_df.withColumn(
    "device_info",
    col("device_info").withField(
        "status",
        when(lower(trim(col("device_info.status"))).isin("n/a", "unknown", "error"), None)
        .otherwise(lower(trim(col("device_info.status"))))
    )
)
iot_final_df.display()

## 7 · Save as silver (idempotent)

`overwrite` replaces the table on every run — re-running the notebook never
duplicates data.

In [ ]:
iot_final_df.write.mode("overwrite").saveAsTable("dev.mini_projects.iot_device_events_silver")

In [ ]:
spark.table("dev.mini_projects.iot_device_events_silver").display()

## Key takeaways

1. **Know your grain before you dedup.** Subset-dedup on an event table silently
   deletes real events — and a matching row count can hide it.
2. **Clean text first, cast second.** `"14,20 EUR"` → `"14.20"` → `decimal(10,2)`.
3. **Multi-format timestamps** → `coalesce(try_to_timestamp × n)`, then *verify*
   nothing became `NULL` silently.
4. **Store UTC** at the silver layer; convert for display only.
5. **Match the JSON shape to the schema type**: fixed keys → `struct`,
   repeated items → `array<struct>`, arbitrary pairs → `map`.
6. **`withField`** cleans inside a struct without restructuring it.
7. **`overwrite`** keeps the pipeline idempotent.